In [4]:
import pandas as pd
import requests
import yaml
import json


# ==============================
# Load YAML
# ==============================

yaml_url = (
    "https://raw.githubusercontent.com/OTRF/OSSEM-DM/main/"
    "use-cases/mitre_attack/techniques_to_events_mapping.yaml"
)

response = requests.get(yaml_url)
response.raise_for_status()

yaml_mapping = yaml.safe_load(response.text)

df = pd.json_normalize(yaml_mapping)


print("Original shape:", df.shape)


Original shape: (11075, 20)


In [5]:
# ==============================
# Universal cleaner
# ==============================

def clean_value(x):

    # Handle None first
    if x is None:
        return None


    # Handle NaN only for scalar values
    if not isinstance(x, (list, dict)):
        try:
            if pd.isna(x):
                return None
        except:
            pass


    # Handle lists
    if isinstance(x, list):

        cleaned = []

        for item in x:

            if isinstance(item, dict):

                cleaned.append(
                    json.dumps(item, sort_keys=True)
                )

            else:

                cleaned.append(
                    str(item)
                )

        return ", ".join(cleaned)


    # Handle dictionaries

    if isinstance(x, dict):

        return json.dumps(
            x,
            sort_keys=True
        )


    # Normal values

    return str(x)


# ==============================
# Apply cleaning
# ==============================

for col in df.columns:

    df[col] = df[col].apply(clean_value)



# ==============================
# Normalize important columns
# ==============================

text_columns = [
    "technique",
    "data_source",
    "data_component",
    "event_name",
    "log_source",
    "channel",
    "event_platform"
]


for col in text_columns:

    if col in df.columns:

        df[col] = (
            df[col]
            .fillna("")
            .str.lower()
            .str.strip()
        )



# ==============================
# Create matching keys
# ==============================


df["event_identity"] = (
    df["event_platform"]
    + "|"
    + df["log_source"]
    + "|"
    + df["event_id"].astype(str)
    + "|"
    + df["event_name"]
)



df["source_identity"] = (
    df["event_platform"]
    + "|"
    + df["data_source"]
    + "|"
    + df["data_component"]
)



# ==============================
# Remove duplicates
# ==============================

clean_mapping = (
    df
    .drop_duplicates()
    .reset_index(drop=True)
)



print(
    "Clean shape:",
    clean_mapping.shape
)



clean_mapping.head()

Clean shape: (11075, 22)


,technique_id,is_subtechnique,technique,tactic,platform,data_source,data_component,relationship_id,name,source,...,event_id,event_name,event_platform,audit_category,audit_sub_category,channel,log_source,filter_in,event_identity,source_identity
0,T1622,False,debugger evasion,"defense-evasion, discovery","Windows, Linux, macOS",process,os api execution,REL-2022-0150,Process executed Api call,process,...,8,createremotethread.,windows,CreateRemoteThread,None,microsoft-windows-sysmon/operational,microsoft-windows-sysmon,None,windows|microsoft-windows-sysmon|8|createremot...,windows|process|os api execution
1,T1622,False,debugger evasion,"defense-evasion, discovery","Windows, Linux, macOS",process,os api execution,REL-2022-0150,Process executed Api call,process,...,DeviceEvents,deviceevents,windows,None,None,,microsoft defender for endpoint,"{""ActionType"": ""CreateRemoteThreadApiCall""}",windows|microsoft defender for endpoint|Device...,windows|process|os api execution
2,T1622,False,debugger evasion,"defense-evasion, discovery","Windows, Linux, macOS",process,process creation,REL-2022-0146,User created Process,user,...,4688,a new process has been created.,windows,Detailed Tracking,Process Creation,security,microsoft-windows-security-auditing,None,windows|microsoft-windows-security-auditing|46...,windows|process|process creation
3,T1622,False,debugger evasion,"defense-evasion, discovery","Windows, Linux, macOS",process,process creation,REL-2022-0146,User created Process,user,...,1,process creation.,windows,ProcessCreate,None,microsoft-windows-sysmon/operational,microsoft-windows-sysmon,None,windows|microsoft-windows-sysmon|1|process cre...,windows|process|process creation
4,T1622,False,debugger evasion,"defense-evasion, discovery","Windows, Linux, macOS",process,process creation,REL-2022-0146,User created Process,user,...,1,process creation.,linux,ProcessCreate,None,,sysmon,None,linux|sysmon|1|process creation.,linux|process|process creation


In [6]:
print(clean_mapping)

      technique_id is_subtechnique              technique  \
0            T1622           False       debugger evasion   
1            T1622           False       debugger evasion   
2            T1622           False       debugger evasion   
3            T1622           False       debugger evasion   
4            T1622           False       debugger evasion   
...            ...             ...                    ...   
11070        T1003            None  os credential dumping   
11071        T1003            None  os credential dumping   
11072        T1003            None  os credential dumping   
11073        T1003            None  os credential dumping   
11074        T1003            None  os credential dumping   

                           tactic               platform       data_source  \
0      defense-evasion, discovery  Windows, Linux, macOS           process   
1      defense-evasion, discovery  Windows, Linux, macOS           process   
2      defense-evasion, discovery

In [11]:
clean_mapping.to_csv("cleand_mapping.csv")

In [7]:
print(clean_mapping.shape)
print("*"*50)
print(clean_mapping.isnull().sum())

(11075, 22)
**************************************************
technique_id             0
is_subtechnique       3044
technique                0
tactic                   0
platform                 0
data_source              0
data_component           0
relationship_id          0
name                     0
source                   0
relationship             0
target                 306
event_id                 0
event_name               0
event_platform           0
audit_category        3981
audit_sub_category    7352
channel                  0
log_source               0
filter_in             7263
event_identity           0
source_identity          0
dtype: int64


In [8]:
"""
ATT&CK Telemetry Evidence Confidence Engine
=============================================

Scores every (technique, telemetry evidence) relationship in a cleaned
ATT&CK/OSSEM mapping dataframe by how DISCRIMINATIVE that evidence is --
i.e. how much it actually narrows down which technique is happening,
versus how much it's shared across many techniques and therefore only
weakly informative on its own.

Model
-----
Let T = set of distinct techniques, E = set of distinct evidence identities
(event_identity: platform|log_source|event_id|event_name), and R the set of
observed (t, e) relationship rows.

For evidence e:
    deg(e) = number of distinct techniques e is linked to in R

Inverse Technique Frequency (specificity), Laplace-smoothed so deg=0 never
divides by zero and very common evidence doesn't hit exactly log(1)=0:

    ITF(e) = log( (|T| + 1) / (deg(e) + 1) )

This is normalized to [0,1] across the whole dataset (itf_norm), giving a
GLOBAL specificity score: is this evidence rare/unique across all of
ATT&CK, or generic?

A second term captures TECHNIQUE-RELATIVE importance: within a single
technique's own evidence set, how much of the available "specificity mass"
does this particular row represent? A technique with very few evidence
types will lean more heavily on each one it has, even if that evidence
type isn't globally rare.

    relative_importance(t,e) = ITF(e) / sum(ITF(e') for e' in evidence(t))

Also normalized to [0,1] (relative_importance_norm).

Final score is a weighted blend (alpha defaults to favoring global
specificity, since that's the stronger signal for cross-technique
disambiguation, which is the whole point of this layer):

    evidence_score(t,e) = alpha * itf_norm(e) + (1-alpha) * relative_importance_norm(t,e)

Nothing here references a specific event ID, log source name, or technique
ID -- every score is derived purely from the co-occurrence structure of
the dataset, so it recalculates correctly as the mapping table grows or
changes.

Explanations are template-generated from the same two components (deg(e)
and relative_importance_norm), not from a lookup table.
"""

import numpy as np
import pandas as pd


def build_evidence_confidence_engine(
    df: pd.DataFrame,
    technique_col: str = "technique_id",
    evidence_col: str = "event_identity",
    alpha: float = 0.7,
) -> pd.DataFrame:
    """
    Enrich a cleaned ATT&CK/OSSEM mapping dataframe with adaptive evidence
    relevance scores and explanations.

    Parameters
    ----------
    df : cleaned mapping dataframe (output of clean_and_prepare)
    technique_col : column identifying the ATT&CK technique (default technique_id,
        which correctly separates sub-techniques like T1059.001 from T1059)
    evidence_col : column identifying a unit of telemetry evidence. Default
        is event_identity (platform|log_source|event_id|event_name), the
        most granular/detection-actionable level. Swap in source_identity
        for a data_source/data_component-level score instead.
    alpha : blend weight between global specificity (itf_norm) and
        technique-relative importance (relative_importance_norm). Higher
        alpha = trust global rarity more; lower alpha = weight a
        technique's own evidence context more.

    Returns
    -------
    A copy of df with these new columns:
        evidence_degree            - # distinct techniques this evidence supports
        evidence_score             - final blended score in [0, 1]
        evidence_confidence_reason - natural-language explanation
    """
    if not 0 <= alpha <= 1:
        raise ValueError("alpha must be between 0 and 1")

    d = df.copy()
    d[technique_col] = d[technique_col].fillna("unknown")
    d[evidence_col] = d[evidence_col].fillna("unknown")

    total_techniques = d[technique_col].nunique()

    # Evidence degree: how many distinct techniques does this evidence link to?
    evidence_degree = d.groupby(evidence_col)[technique_col].transform("nunique")
    d["evidence_degree"] = evidence_degree

    # Inverse Technique Frequency (global specificity), Laplace-smoothed
    d["_itf_raw"] = np.log((total_techniques + 1) / (d["evidence_degree"] + 1))

    itf_min, itf_max = d["_itf_raw"].min(), d["_itf_raw"].max()
    d["_itf_norm"] = (d["_itf_raw"] - itf_min) / (itf_max - itf_min + 1e-9)

    # Technique-relative importance: share of this technique's own
    # specificity budget that this evidence row accounts for
    tech_itf_sum = d.groupby(technique_col)["_itf_raw"].transform("sum")
    d["_relative_importance"] = d["_itf_raw"] / (tech_itf_sum + 1e-9)

    ri_min, ri_max = d["_relative_importance"].min(), d["_relative_importance"].max()
    d["_relative_importance_norm"] = (
        (d["_relative_importance"] - ri_min) / (ri_max - ri_min + 1e-9)
    )

    d["evidence_score"] = (
        alpha * d["_itf_norm"] + (1 - alpha) * d["_relative_importance_norm"]
    ).round(4)

    d["evidence_confidence_reason"] = d.apply(
        lambda row: _explain(row, technique_col), axis=1
    )

    d = d.drop(columns=["_itf_raw", "_itf_norm", "_relative_importance", "_relative_importance_norm"])
    return d


def _explain(row, technique_col) -> str:
    deg = int(row["evidence_degree"])
    score = row["evidence_score"]
    other = deg - 1

    if deg <= 1:
        specificity_text = f"unique to {row[technique_col]} across the dataset"
    elif deg <= 5:
        specificity_text = f"shared with only {other} other technique(s), so still fairly specific"
    elif deg <= 20:
        specificity_text = f"shared with {other} other techniques, giving it moderate specificity"
    else:
        specificity_text = f"shared with {other} other techniques, so it has low discriminative power on its own"

    ri = row["_relative_importance_norm"]
    if ri > 0.66:
        weight_text = "; it's also one of the primary evidence sources available for this technique"
    elif ri > 0.33:
        weight_text = "; it contributes a moderate share of the available evidence for this technique"
    else:
        weight_text = "; relative to other evidence for this technique, it's a comparatively minor source"

    return f"Evidence score {score:.2f}: this telemetry is {specificity_text}{weight_text}."


if __name__ == "__main__":
    pass

In [9]:
df_scored = build_evidence_confidence_engine(clean_mapping)

In [11]:
df_scored.to_csv('clean_mapping_dataset.csv',  index=False)